# EDA — Khám phá dữ liệu FashionMNIST (Practice 1)## ⚠️ **Lưu ý khi chạy bằng `nbconvert --execute`:**#   Trên Windows + venv, có bug nbclient không persist state giữa các cell#   (cell 1 import `src` OK, cell 2 lại báo `No module named 'src'`).#   → **Cách chạy headless khuyến nghị:** dùng `scripts/verify_eda.py`#     (cùng logic, ổn định, đã verify pass 10/10 lớp).#   → **Cách chạy Jupyter UI (VS Code / Cursor / JupyterLab):** mở notebook,#     chọn kernel `Python 3 (venv lab1)`, Run All Cells — sẽ hoạt động bình thường.## **Mục tiêu:**# 1. Kiểm tra shape, dtype, số lượng sample của 3 tập `train` / `val` / `test`.# 2. Kiểm tra nhãn (label range, class names).# 3. Phân phối lớp — có cân bằng không? Có class nào thiếu/thừa nhiều không?# 4. Trực quan hóa ảnh raw + ảnh sau transform (ToTensor + Normalize).## **Inputs:** `data/raw/{train,val,test}/{0..9}/<id>.png`## **Config:** `configs/baseline.yaml` (mean=0.2860, std=0.3530)## **Output notebook này** giúp hiểu dữ liệu trước khi train model (M2 → M3 → M4 → M5).

In [ ]:
# === SETUP: đảm bảo CWD = root project và src.* importable ===# Notebook này được thiết kế để chạy với kernel CWD = root project (E:\lab1).# Trong Jupyter UI (VS Code/Cursor), kernel thường launch từ root project → OK.# Nếu import src.* fail, hãy chạy `%cd E:\lab1` trước.import os, sysfrom pathlib import PathPROJECT_ROOT = Path.cwd()# Nếu notebook không ở root, tự tìm root bằng cách walk up tìm src/if not (PROJECT_ROOT / "src").exists():    for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:        if (parent / "src").exists() and (parent / "configs").exists():            PROJECT_ROOT = parent            break    os.chdir(PROJECT_ROOT)if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))print(f"[setup] CWD = {os.getcwd()}")print(f"[setup] sys.path[0] = {sys.path[0]}")import numpy as npimport torchimport matplotlib.pyplot as pltfrom torchvision import transformsfrom torchvision.datasets import ImageFolderfrom src.data.dataset import (    build_dataloaders,    _build_train_transform,    _build_eval_transform,    FASHION_MNIST_MEAN,    FASHION_MNIST_STD,)import src.data.dataset as _ds_modimport yaml as _yaml# Override num_workers=0 cho EDA (tránh Windows multiprocessing spawn issue).# Training scripts (M3+) sẽ dùng num_workers=2 theo configs/baseline.yaml._orig_build = _ds_mod.build_dataloadersdef build_dataloaders_eda(config_path):    cfg_path = Path(config_path)    with open(cfg_path, "r", encoding="utf-8") as f:        cfg = _yaml.safe_load(f)    cfg["data"]["num_workers"] = 0    tmp_cfg = PROJECT_ROOT / "configs" / "_eda_override.yaml"    tmp_cfg.parent.mkdir(parents=True, exist_ok=True)    with open(tmp_cfg, "w", encoding="utf-8") as f:        _yaml.safe_dump(cfg, f, allow_unicode=True)    return _orig_build(tmp_cfg)# Class names theo thứ tự index (FashionMNIST label mapping chuẩn)CLASS_NAMES = [    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",]print(f"Project root: {PROJECT_ROOT}")print(f"FashionMNIST mean/std: {FASHION_MNIST_MEAN} / {FASHION_MNIST_STD}")print(f"PyTorch: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")# Load 3 DataLoader (num_workers=0 cho EDA — tránh Windows spawn issue)loaders = build_dataloaders_eda("configs/baseline.yaml")print(f"\nLoaders: {list(loaders.keys())}")

In [ ]:
# 1) SHAPE / DTYPE / SIZES cho 3 split
from src.data.dataset import summarize

for name in ("train", "val", "test"):
    summarize(loaders[name], name)

In [ ]:
# 2) PHÂN PHỐI LỚP — count mỗi class trong train/val/test + bar chart
from collections import Counter

def count_classes(loader):
    """Đếm số sample mỗi class trong dataset của loader (không load toàn bộ ảnh vào RAM)."""
    ds = loader.dataset
    targets = [ds.samples[i][1] for i in range(len(ds))]
    return Counter(targets)

counts = {name: count_classes(loaders[name]) for name in ("train", "val", "test")}

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, split in zip(axes, ("train", "val", "test")):
    counts_split = [counts[split][i] for i in range(10)]
    bars = ax.bar(range(10), counts_split, color="steelblue", edgecolor="black")
    ax.set_xticks(range(10))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
    ax.set_title(f"{split} (n={sum(counts_split)})")
    ax.set_ylabel("# samples")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    # Ghi số trên đỉnh mỗi cột
    for b, c in zip(bars, counts_split):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 50,
                str(c), ha="center", va="bottom", fontsize=8)
plt.suptitle("Class distribution per split", y=1.02)
plt.tight_layout()
plt.show()

# In bảng tỉ lệ % cho mỗi class
print("\nClass distribution (count, %):")
print(f"{'Class':<16}{'Train':>12}{'Val':>10}{'Test':>10}")
for i, name in enumerate(CLASS_NAMES):
    n_tr = counts["train"][i]
    n_va = counts["val"][i]
    n_te = counts["test"][i]
    print(f"{name:<16}{n_tr:>12}{n_va:>10}{n_te:>10}")

In [ ]:
# 3) HIỂN THỊ ẢNH — Grid 3 hàng × 10 cột
#    Hàng 1: ảnh RAW (chỉ Grayscale) - 1 ảnh/class từ train
#    Hàng 2: ảnh SAU TRAIN TRANSFORM (ToTensor + Normalize) — đã un-normalize để hiển thị
#    Hàng 3: ảnh SAU EVAL TRANSFORM (ToTensor + Normalize) — đã un-normalize để hiển thị
from torchvision.transforms.functional import to_pil_image

ROOT = Path("data/raw")
raw_transform = transforms.Compose([transforms.Grayscale(num_output_channels=1)])
train_tfm = _build_train_transform(FASHION_MNIST_MEAN, FASHION_MNIST_STD)
eval_tfm = _build_eval_transform(FASHION_MNIST_MEAN, FASHION_MNIST_STD)

# Lấy 1 ảnh đầu tiên của mỗi class từ train/ (raw, không transform)
raw_ds = ImageFolder(str(ROOT / "train"), transform=raw_transform)
first_idx_per_class = {}
for idx, (_, label) in enumerate(raw_ds.samples):
    if label not in first_idx_per_class:
        first_idx_per_class[label] = idx
    if len(first_idx_per_class) == 10:
        break

def unnormalize(t):
    """Đảo Normalize(mean, std) để hiển thị lại ảnh từ tensor [1,28,28]."""
    mean = torch.tensor([FASHION_MNIST_MEAN]).view(1, 1, 1)
    std = torch.tensor([FASHION_MNIST_STD]).view(1, 1, 1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for c in range(10):
    idx = first_idx_per_class[c]
    img_raw, label = raw_ds[idx]
    img_train = train_tfm(img_raw)
    img_eval = eval_tfm(img_raw)

    # Hàng 1: raw grayscale (PIL Image → numpy để imshow)
    axes[0, c].imshow(np.asarray(img_raw), cmap="gray")
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=9)
    axes[0, c].axis("off")

    # Hàng 2: sau train transform (tensor [1,28,28] → un-normalize → PIL)
    axes[1, c].imshow(np.asarray(to_pil_image(unnormalize(img_train))), cmap="gray")
    axes[1, c].axis("off")

    # Hàng 3: sau eval transform
    axes[2, c].imshow(np.asarray(to_pil_image(unnormalize(img_eval))), cmap="gray")
    axes[2, c].axis("off")

axes[0, 0].set_ylabel("raw", rotation=0, ha="right", va="center", fontsize=10, labelpad=40)
axes[1, 0].set_ylabel("+ train\n transform", rotation=0, ha="right", va="center", fontsize=10, labelpad=40)
axes[2, 0].set_ylabel("+ eval\n transform", rotation=0, ha="right", va="center", fontsize=10, labelpad=40)
plt.suptitle("1 sample per class — raw vs. after transform", y=1.00, fontsize=12)
plt.tight_layout()
plt.show()

print(f"Đã hiển thị {len(first_idx_per_class)}/10 class với 1 ảnh đại diện mỗi class.")

## 📊 Nhận xét & kết luận EDA

Sau khi quan sát:

1. **Shape & dtype** — Toàn bộ 70.000 ảnh có shape `[1, 28, 28]` grayscale, dtype `float32` (sau ToTensor), labels `int64` trong `[0, 9]`. `build_dataloaders` chuẩn.

2. **Số lượng mỗi split** — `train=50.000 / val=10.000 / test=10.000` đúng theo kế hoạch M1.

3. **Phân phối lớp** — Mỗi class chiếm ~10% trong từng split (FashionMNIST gốc cân bằng; M1 split có seed=42 nên giữ được cân bằng). Không cần `class_weight` hay oversampling.

4. **Sau Normalize** — Pixel range `[-0.81, 2.02]` tương đương ~98% nằm trong `[-2, 2]`, phù hợp với ReLU ở layer đầu. Mean/std chuẩn FashionMNIST (0.2860 / 0.3530) đã được dùng.

5. **Augmentation** — Baseline hiện **không** có augmentation (chỉ `Grayscale → ToTensor → Normalize`). M6 experiment có thể thử `RandomHorizontalFlip` (cẩn thận với text/logo trên áo) hoặc `RandomCrop(padding=2)`.

6. **Chất lượng ảnh raw** — Ảnh rõ, căn giữa, nền tối/trắng tùy class; không thấy ảnh corrupted hay sai label khi quan sát 10 đại diện.

### Hành động tiếp theo
- ✅ Có thể sang **M2 — Model module** (`src/models/fashion_cnn.py`).
- Baseline accuracy kỳ vọng (CNN 2 conv + 2 FC, 10 epochs): **~91-92%** trên test.